# AIVORA — Intelligent Resume-to-Job Matching (Google Colab)
Run the cells **from top to bottom** (Runtime → Run all works too, except the last cell which keeps running).

1. **Get the code** – clone from GitHub, or upload `AIVORA.zip`
2. **Install** dependencies
3. **Test** the skill extractor
4. **Demo** – rank the sample resumes in the notebook (no browser needed)
5. **Web app** – start the Streamlit interface through a temporary public link

> Scores support human review; they are not hiring decisions. Do not upload real/private resumes to a public tunnel.

In [1]:
# 1. Get the code -------------------------------------------------------------
# Option A: paste your GitHub repository URL, e.g. "https://github.com/<user>/AIVORA.git"
# Option B: leave it empty and upload AIVORA.zip when asked.
REPO_URL = ""

import os, subprocess, zipfile
os.chdir("/content")
if REPO_URL.strip():
    subprocess.run(["rm", "-rf", "AIVORA"])
    subprocess.run(["git", "clone", REPO_URL.strip(), "AIVORA"], check=True)
else:
    from google.colab import files
    uploaded = files.upload()                       # choose AIVORA.zip
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall("/content")
# find the project folder (the one that contains app.py)
for root, dirs, fs in os.walk("/content"):
    if "app.py" in fs and "modules" in dirs:
        os.chdir(root); break
print("Working directory:", os.getcwd())
print(os.listdir("."))

Saving AIVORA.zip to AIVORA.zip
Working directory: /content/AIVORA
['modules', 'notebooks', '.gitignore', 'data', 'conftest.py', 'demo.py', 'README.md', 'requirements.txt', 'tests', 'app.py']


In [2]:
# 2. Install dependencies (about 1-3 minutes) --------------------------------
!pip -q install -r requirements.txt

In [3]:
# 3. Run the unit tests -------------------------------------------------------
!python -m pytest -q tests

....                                                                     [100%]
4 passed in 6.02s


In [5]:
# 4. Demo: rank the sample resumes against the sample job description --------
# The first run downloads the Sentence-BERT model (all-MiniLM-L6-v2).
!python demo.py

modules.json: 100% 349/349 [00:00<00:00, 1.30MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 429kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 24.0MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 180kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.71MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:01<00:00, 114MB/s, 7.53MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 74.1MB/s, 8.18MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 79.2MB/s, 8.84MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 4984.58it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.19MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 10.5MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 39.8MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 413kB/s]
config.json: 100% 190/190 [00:00<00:00, 594kB/s]
Rank Resume              Overall  Semantic  Coverage
1    resume_a.txt           80.3      67.2     100.

In [ ]:
# 5a. Start the Streamlit app in the background ------------------------------
import subprocess, time
subprocess.run("pkill -f streamlit", shell=True)
log = open("/content/streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501",
                  "--server.address", "0.0.0.0", "--server.headless", "true"],
                 stdout=log, stderr=log)
time.sleep(10)
print(open("/content/streamlit.log").read())

In [ ]:
# 5b. Create a temporary public link ------------------------------------------
# Keep this cell running while you use the app. Stop it (square button) to close the link.
# When the page asks for a "Tunnel Password", use the IP address printed below.
!npm install -g localtunnel --silent
import urllib.request
print("Tunnel password (public IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!npx localtunnel --port 8501

Tunnel password (public IP): 34.106.51.213
⠙⠹⠸⠼⠴your url is: https://tangy-houses-smile.loca.lt


### Using the app
* **Recruiter workspace:** paste the job description (try `data/sample_job_description.txt`), upload several resumes (try the files in `data/sample_resumes/`), press **Analyze and rank candidates**, and download the CSV shortlist.
* **Candidate workspace:** paste a job description, upload one resume, press **Generate my skill-gap report**.